# Day 4 Lab: Delta Lake Pipeline in Databricks

## Objectives
- Understand Delta Lake
- Perform MERGE (Upserts)
- Use Time Travel
- Build reliable pipelines


In [0]:
# Upload the orders csv file into DBFS through the UI
# mention appropriate catalog, schema and volume names
# orders_df = spark.read.option('header', True).csv('dbfs:/Volumes/<catalog_name>/<schema_name>/<volumne_name>/orders.csv')

# Example
orders_df = spark.read.option('header', True).csv('dbfs:/Volumes/dbcuser01/ingesting_data/myfiles/orders.csv')

display(orders_df)

In [0]:
# Update with proper catalog, schema and volume names 
# orders_df.write.format('delta').mode('overwrite').save('/Volumes/<catalog_name>/<schema_name>/<volume_name>/orders_delta1')

orders_df.write.format('delta').mode('overwrite').save('/Volumes/dbcuser01/ingesting_data/myfiles/orders_delta1')

In [0]:
# Update with proper catalog, schema and volume names 
# delta_df = spark.read.format('delta').load('/Volumes/<catalog_name>/<schema_name>/<volume_name>/orders_delta1')

delta_df = spark.read.format('delta').load('/Volumes/dbcuser01/ingesting_data/myfiles/orders_delta1')


display(delta_df)

In [0]:
new_data = [
 (7,101,'2024-01-06',300,'completed'),
 (2,102,'2024-01-02',350,'completed')
]
new_df = spark.createDataFrame(new_data, ['order_id','customer_id','order_date','amount','status'])
display(new_df)

In [0]:
from delta.tables import DeltaTable

# update with proper catalog, schema and volume names
# delta_table = DeltaTable.forPath(spark, '/Volumes/<catalog_name>/<schema_name>/<volume_name>/orders_delta1')

delta_table = DeltaTable.forPath(spark, '/Volumes/dbcuser01/ingesting_data/myfiles/orders_delta1')

delta_table.alias('target').merge(
    new_df.alias('source'),
    'target.order_id = source.order_id'
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

In [0]:
display(delta_table.toDF())

In [0]:
spark.read.format('delta').option('versionAsOf', 1).load('/Volumes/dbcuser01/ingesting_data/myfiles/orders_delta1')

In [0]:
bad_data = [(8, 'invalid_customer', 'bad_date', 'wrong_amount', 'completed')]
bad_df = spark.createDataFrame(bad_data)

# update with proper catalog, schema and volume names
# bad_df.write.format('delta').option('mergeSchema', 'true').mode('append').save('/Volumes/<catalog_name>/<schema_name>/<volume_name>/orders_delta')

bad_df.write.format('delta').option('mergeSchema', 'true').mode('append').save('/Volumes/dbcuser01/ingesting_data/myfiles/orders_delta')

In [0]:
# update with proper catalog, schema and volume names
# spark.sql("OPTIMIZE delta.`/Volumes/<catalog_name>/<schema_name>/<volume_name>/orders_delta1`")

spark.sql("OPTIMIZE delta.`/Volumes/dbcuser01/ingesting_data/myfiles/orders_delta1`")